# Week 44, Day 3: Choosing actions: bandits, exploration and fitting choices

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Regret of four strategies

Complete choose(Q, N, t) for each policy: greedy (argmax with tiny random tie-breaking), epsilon-greedy, softmax (sample by comparing the cumulative probabilities with a uniform number) and UCB (untried arms first). All 500 runs are simulated in parallel as rows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2006)
K, T, R = 10, 1000, 500
mu = r.normal(size=(R, K)); best = mu.max(1); rows = np.arange(R)

def choose(policy, par, Q, N, t):
    if policy == "greedy":
        return np.argmax(Q + 1e-9 * r.random((R, K)), 1)
    if policy == "epsilon":
        a = np.argmax(Q + 1e-9 * r.random((R, K)), 1)
        ...
        return a
    if policy == "softmax":
        p = np.exp(par * (Q - Q.max(1, keepdims=True))); p /= p.sum(1, keepdims=True)
        return ...
    if policy == "ucb":
        bonus = np.where(N == 0, 1e9, par * np.sqrt(np.log(t + 1) / np.maximum(N, 1)))
        return ...

def run(policy, par):
    Q, N, reg = np.zeros((R, K)), np.zeros((R, K)), np.zeros(T)
    for t in range(T):
        a = choose(policy, par, Q, N, t)
        rew = mu[rows, a] + r.normal(size=R)
        N[rows, a] += 1; Q[rows, a] += (rew - Q[rows, a]) / N[rows, a]
        reg[t] = (best - mu[rows, a]).mean()
    return reg

final, total = {}, {}
for name, pol, par in [("greedy", "greedy", 0), ("epsilon 0.1", "epsilon", 0.1), ("softmax 3", "softmax", 3.0), ("UCB 1", "ucb", 1.0)]:
    reg = run(pol, par); final[name], total[name] = reg[-100:].mean(), reg.sum()
    plt.plot(np.cumsum(reg), label=name)
    print(f"{name:12s} regret per step at the end {final[name]:.3f}, total {total[name]:.0f}")
plt.xlabel("step"); plt.ylabel("cumulative regret"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_greedy():
    "greedy has the highest final regret per step"
    assert final["greedy"] > max(final["epsilon 0.1"], final["softmax 3"], final["UCB 1"])
def test_ucb():
    "UCB has the lowest total regret"
    assert total["UCB 1"] < min(total["epsilon 0.1"], total["softmax 3"], total["greedy"])

test_greedy()
test_ucb()
print("All checks passed")

### Exercise: Log-likelihood on a grid

Write nll(alpha, beta, ch, rw), the negative log-likelihood of the choices under the softmax Q-learning model (use np.logaddexp for stability). Then evaluate it on a grid of alpha and beta and find the minimum.

In [ ]:
def simulate(alpha, beta, T=300, seed=1):
    g = np.random.default_rng(seed); p_rew = np.array([0.7, 0.3]); Q = np.zeros(2); ch, rw = [], []
    for t in range(T):
        if t == T // 2:
            p_rew = p_rew[::-1]
        p1 = 1 / (1 + np.exp(-beta * (Q[1] - Q[0])))
        a = int(g.random() < p1); rr = float(g.random() < p_rew[a])
        Q[a] += alpha * (rr - Q[a]); ch.append(a); rw.append(rr)
    return np.array(ch), np.array(rw)

def nll(alpha, beta, ch, rw):
    Q = np.zeros(2); L = 0.0
    for a, rr in zip(ch, rw):
        ...
    return L

ch, rw = simulate(0.3, 5.0)
A = np.linspace(0.05, 0.95, 19); B = np.linspace(0.5, 10, 20)
L = np.array([[nll(a, b, ch, rw) for b in B] for a in A])
i, j = np.unravel_index(L.argmin(), L.shape); a_hat, b_hat = A[i], B[j]
print(f"NLL at the truth {nll(0.3, 5.0, ch, rw):.1f}, random chooser {len(ch) * np.log(2):.1f}; grid minimum alpha {a_hat:.2f}, beta {b_hat:.1f}")
plt.contourf(B, A, L, 30); plt.plot(5, 0.3, "w*", ms=12); plt.plot(b_hat, a_hat, "rx"); plt.xlabel("beta"); plt.ylabel("alpha"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_nll():
    "the negative log-likelihood at the true parameters is below that of a random chooser"
    assert nll(0.3, 5.0, ch, rw) < len(ch) * np.log(2)
def test_grid():
    "the grid minimum is within 0.1 of the true alpha and 1.5 of the true beta"
    assert abs(a_hat - 0.3) <= 0.1 and abs(b_hat - 5.0) <= 1.5

test_nll()
test_grid()
print("All checks passed")

### Exercise: Parameter recovery

Simulate 20 participants with alpha drawn from 0.1 to 0.6 and beta from 2 to 8, each with its own seed; fit each with scipy.optimize.minimize (method L-BFGS-B, bounds alpha 0.01 to 0.99 and beta 0.1 to 20, starting at 0.5 and 3) and collect the estimates.

In [ ]:
from scipy.optimize import minimize
g = np.random.default_rng(50)
true, fit = [], []
for s in range(20):
    a0, b0 = g.uniform(0.1, 0.6), g.uniform(2, 8)
    c, w = simulate(a0, b0, seed=100 + s)
    res = ...
    true.append((a0, b0)); fit.append(res.x)
true, fit = np.array(true), np.array(fit)
print(f"recovery: r(alpha) = {np.corrcoef(true[:, 0], fit[:, 0])[0, 1]:.2f}, r(beta) = {np.corrcoef(true[:, 1], fit[:, 1])[0, 1]:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
for k, nm in enumerate(["alpha", "beta"]):
    ax[k].plot(true[:, k], fit[:, k], "o"); lim = [true[:, k].min(), true[:, k].max()]; ax[k].plot(lim, lim, "k:"); ax[k].set_xlabel("true " + nm); ax[k].set_ylabel("fitted " + nm)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_alpha():
    "fitted learning rates correlate with true ones above 0.6"
    assert np.corrcoef(true[:, 0], fit[:, 0])[0, 1] > 0.6
def test_beta():
    "fitted inverse temperatures correlate with true ones above 0.6"
    assert np.corrcoef(true[:, 1], fit[:, 1])[0, 1] > 0.6

test_alpha()
test_beta()
print("All checks passed")